# Tutorial 150 — Inspect a KíLín execution trace

[Tutorial 140](./140_project_chat.ipynb) used Project Chat as a user-facing
workflow. This tutorial opens the same architecture from the opposite
direction: it follows one operation at a time through a **complete execution trace**.

You will connect source preparation, model selection, indexing, question
embedding, retrieval and ranking, exact LLM input, generation, saved evidence,
and operation timings to the trace sections produced by the corresponding
function calls.

The goal is not to read a large JSON document from top to bottom. Instead, use
the trace as an audit trail: start from the operation you want to understand and
expand only the relevant stage.

**Before you begin:** See the [tutorial guide](./README.md) for setup, model downloads, learning paths, and workspace safety.


## 1. Prepare KíLín

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "kilin").is_dir():
    root = root.parent

print(f"KíLín root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import kilin


## 2. Choose the project and create one shared trace

This tutorial uses one PDF and a new workspace so that all operations are recorded in a chronological trace.

Enabled tracing keeps retrieved-text previews, the rendered model prompt,
token IDs, generated output, and detailed operation records. The complete index
is intentionally not duplicated into the trace. Print the source, workspace,
and trace paths first so every later event can be tied to concrete files.

The supplementary paper is used from `_si_article_/literature_repo/papers/`
when present. The bundled `tests/fixtures/` PDF is a fallback for source ZIPs.
New tutorial workspaces disable model downloads by default; change the
`ALLOW_DOWNLOAD` switch below if you explicitly want to download weights.
The notebook chooses a fresh numbered workspace under `sandbox/` instead of
overwriting an earlier run. To resume a specific workspace, use
`kilin.open_project(path)` explicitly.

**Privacy:** Traces may contain complete prompts, passages, model output,
tokens, and local paths. Keep workspaces private; a trace file may also hold
resumable conversation state, so do not delete it without checking first.


In [ ]:
import json
from html import escape
from IPython.display import HTML, Markdown, display

PROJECT_TYPE_ID = "literature"
article_root = root / "_si_article_" / "literature_repo"
article_paper = article_root / "papers" / "mijalkov25computational.pdf"
fixture_paper = root / "tests" / "fixtures" / article_paper.name

if article_paper.is_file():
    source_directory = article_root.resolve()
    SELECTED_FILE = "papers/mijalkov25computational.pdf"
elif fixture_paper.is_file():
    source_directory = fixture_paper.parent.resolve()
    SELECTED_FILE = fixture_paper.name
else:
    raise FileNotFoundError(
        "Add the article corpus or the bundled PDF fixture."
    )

workspace_parent = root / "sandbox"
workspace_parent.mkdir(parents=True, exist_ok=True)
workspace_name = "tutorial_150_single_pdf"
workspace_directory = workspace_parent / workspace_name
suffix = 2
while workspace_directory.exists():
    workspace_directory = workspace_parent / f"{workspace_name}_{suffix}"
    suffix += 1
trace_directory = workspace_directory / "logs" / "traces"

if "session" in globals():
    session.close()

if "trace" in globals():
    trace.close()

project = kilin.prepare_sources_workspace(
    [], PROJECT_TYPE_ID, workspace_directory
)

from configparser import ConfigParser

ALLOW_DOWNLOAD = False
settings = ConfigParser()
settings.read(project.root / "profile.ini", encoding="utf-8")
settings["models"]["allow_download"] = str(ALLOW_DOWNLOAD).lower()
with (project.root / "profile.ini").open("w", encoding="utf-8") as f:
    settings.write(f)
project = kilin.open_project(project.root)

trace = kilin.TraceLogger(
    kilin.TraceConfig(
        enabled=True,
    ),
    default_directory=trace_directory,
)
trace_path = trace.readable_path

print(f"Included source: {source_directory}")
print(f"Selected file: {SELECTED_FILE}")
print(f"Managed workspace: {workspace_directory}")
print(f"Trace file: {trace_path}")


### Trace inspection helpers

Every helper reloads the JSON file from disk, so the next display always reflects events emitted by the cell that just ran. The reported line is the section's current starting line; it may move as later events are added.

In [ ]:
def read_trace() -> dict:
    return json.loads(trace_path.read_text(encoding="utf-8"))


def trace_line(key: str) -> int | None:
    needle = json.dumps(key, ensure_ascii=False)
    for number, line in enumerate(
        trace_path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if needle in line:
            return number
    return None


def named_session(prefix: str) -> tuple[str, dict]:
    matches = [
        (key, value)
        for key, value in read_trace()["sessions"].items()
        if key.startswith(prefix)
    ]
    if not matches:
        raise KeyError(f"No trace session starts with {prefix!r}.")
    return matches[-1]


TRACE_TREE_STYLE = """
<style>
.trace-tree {
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
  font-size: 13px;
}
.trace-tree details {margin: 0.3rem 0 0.3rem 1rem;}
.trace-tree summary {cursor: pointer; line-height: 1.5;}
.trace-tree summary code {font-weight: 650;}
.trace-tree .kind {color: #68757d; margin-left: 0.5rem;}
.trace-tree .scalar {margin: 0.25rem 0 0.25rem 2rem;}
.trace-tree pre {
  white-space: pre-wrap;
  overflow-wrap: anywhere;
  max-height: 28rem;
  overflow: auto;
  margin: 0.4rem 0 0.6rem 1.5rem;
  padding: 0.7rem;
  border-left: 3px solid #b8c9c0;
}
</style>
"""


def trace_tree(label: str, value: object, depth: int = 0) -> str:
    safe_label = escape(str(label))
    if isinstance(value, dict):
        children = "".join(
            trace_tree(str(key), item, depth + 1)
            for key, item in value.items()
        )
        opened = " open" if depth == 0 else ""
        return (
            f"<details{opened}><summary><code>{safe_label}</code>"
            f"<span class='kind'>object · {len(value)} fields</span></summary>"
            f"{children}</details>"
        )
    if isinstance(value, list):
        body = escape(json.dumps(value, indent=2, ensure_ascii=False))
        return (
            f"<details><summary><code>{safe_label}</code>"
            f"<span class='kind'>list · {len(value)} items</span></summary>"
            f"<pre>{body}</pre></details>"
        )
    rendered = json.dumps(value, ensure_ascii=False)
    if isinstance(value, str) and len(value) > 120:
        return (
            f"<details><summary><code>{safe_label}</code>"
            f"<span class='kind'>text · {len(value):,} characters"
            "</span></summary>"
            f"<pre>{escape(value)}</pre></details>"
        )
    return (
        f"<div class='scalar'><code>{safe_label}</code>: "
        f"{escape(rendered)}</div>"
    )


def display_trace_tree(label: str, value: object) -> None:
    tree = trace_tree(label, value)
    display(HTML(TRACE_TREE_STYLE + f"<div class='trace-tree'>{tree}</div>"))


def show_session(prefix: str) -> None:
    key, value = named_session(prefix)
    location = (
        f"**Trace:** `{trace_path}`  \n"
        f"**Starts at current line:** `{trace_line(key)}`  \n"
        f"**Section:** `{key}`"
    )
    display(Markdown(location))
    display_trace_tree(key, value)


def latest_turn() -> tuple[str, dict]:
    _, conversation = named_session("05_conversations")
    turns = conversation.get("turns", {})
    if not turns:
        raise RuntimeError("Ask a question before inspecting a turn.")
    key = list(turns)[-1]
    return key, turns[key]


def show_turn_stage(stage_key: str) -> None:
    turn_key, turn = latest_turn()
    stage = turn["stages"][stage_key]
    location = (
        f"**Turn:** `{turn_key}`  \n"
        f"**Stage:** `{stage_key}`  \n"
        f"**Starts at current line:** `{trace_line(stage_key)}`"
    )
    display(Markdown(location))
    display_trace_tree(stage_key, stage)


## 3. Prepare the included files

This cell asks KíLín to copy exactly the selected PDF into a managed project. In `02_project`, inspect the requested project type and paths, the copied-file operation, and the completion status. This answers: **What did I ask KíLín to prepare, and what did it actually copy?**

In [ ]:
project = kilin.add_project_sources(
    project.root,
    [source_directory],
    selected_files={str(source_directory): [SELECTED_FILE]},
    trace=trace,
)

print(f"Prepared project: {project.root}")
print(f"Copied sources: {project.source}")


### Resolve the models before opening the session

The project profile stores separate chat and embedding model IDs for MLX and
PyTorch. KíLín first resolves the requested backend and then selects the
corresponding pair. Printing all four values here makes model selection explicit
before the trace records model loading.


In [ ]:
SELECTED_BACKEND = kilin.choose_backend(project.profile.backend)
EMBEDDING_MODEL = project.profile.embedding_model(SELECTED_BACKEND)
CHAT_MODEL = project.profile.chat_model(SELECTED_BACKEND)

print(f"Requested backend: {project.profile.backend}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Downloads allowed: {project.profile.allow_download}")


The `02_project` session now shows `workspace.prepare.started`, each copied
source, and `workspace.prepare.completed`. Expand only the operation you want to
inspect. The timestamps and sequence numbers show the real execution order; the
operation details show the values used by the code.


In [ ]:
show_session("02_project")


## 4. Open the RAG session and prepare retrieval

Opening the session loads the embedding model, checks the saved source manifest
and index settings, and either reuses or rebuilds the retrieval index. The same
`TraceLogger` is passed into `RAGSession`, so this remains one chronological
file.

The previous cell already showed the requested backend, resolved backend,
embedding model, and chat model. Compare those choices with the trace. Here,
`Index status` tells you whether an index was reused or rebuilt, `Indexed
chunks` tells you how many searchable pieces were created, and `Backend`
confirms which runtime executed the models.


In [ ]:
session = kilin.RAGSession(
    project.root,
    trace=trace,
    conversation_path=trace_path,
)

print(f"Index status: {session.index_status}")
print(f"Indexed chunks: {len(session.chunks):,}")
print(f"Backend: {session.backend}")


### Source reading and chunking events

`03_sources` records scanning, PDF text extraction, and chunking. Read it as:
selected file → extracted text → searchable chunks. Inspect the file path,
extraction result, chunk settings, and final chunk count. This answers: What did
KíLín read, and how was it divided into searchable evidence?


In [ ]:
show_session("03_sources")


### Indexing events

`04_indexing` shows how the chunks become a searchable index. Read it as: load
embedding model → validate or build index → save index. Inspect the
model/backend, whether validation succeeded, the input/output chunk counts, the
saved index path, and each operation's status. Per-batch document text, token
IDs, masks, and vectors are intentionally omitted. `operation_duration_ms`
means how long that specific named operation took.


In [ ]:
show_session("04_indexing")


## 5. Ask one question

One call to `session.ask()` creates one numbered turn. It embeds the question,
retrieves and ranks chunks, builds the exact model input, calls the LLM once,
saves the answer and evidence, and commits resumable state.

Before opening the trace, read the displayed question, answer, and evidence.
These are the human-facing result. The following numbered stages explain how
KíLín produced that result, in the same order as the real calls.


In [ ]:
QUESTION = (
    "What is computational memory capacity, and what is it used to study?"
)
answer = session.ask(QUESTION)
latest = session.turns[-1]

display(Markdown(f"### Question\n{QUESTION}\n\n### Answer\n{answer}"))
display(Markdown("### Evidence supplied to the LLM"))
for item in latest.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    evidence = (
        f"**[{item.citation}] `{location}`**\n\n"
        f"```text\n{item.text}\n```"
    )
    display(Markdown(evidence))


## 6. Inspect the question event

`01_question` is the starting state for this turn. Inspect the exact question,
whether this is a new chat, how much earlier conversation history is included,
and the final settings used for this turn. This answers: What did the user ask,
and under which settings did KíLín begin?


In [ ]:
show_turn_stage("01_question")


## 7. Inspect question embedding

`02_embedding` records how the question becomes a search query. Inspect the
formatted query preview, embedding model/backend, input token count, vector
dimensions, status, and duration. The resulting vector is used only to compare
the question with indexed chunks; it is not sent to the answer-generating LLM.
This answers: How did KíLín represent the question for semantic search?


In [ ]:
show_turn_stage("02_embedding")


## 8. Inspect retrieval and ranking

`03_retrieval` shows how candidate chunks become the evidence supplied to the
LLM. Follow its nested operations in time: retrieval channels find candidates,
ranking assigns scores and order, context expansion may add neighboring chunks,
and evidence finalization chooses the numbered sources. Inspect each chunk's
path, line range, score, channels, rank, and compact `text_preview`. This
answers: Which project text was found, why was it ranked here, and which pieces
were finally selected?


In [ ]:
show_turn_stage("03_retrieval")


### Read the final hybrid ranking directly from the trace

This cell does not use `session.turns`; it reloads the trace and prints the compact text preview stored under `retrieval.hybrid.ranked`. The complete evidence remains visible in the answer cell and exact LLM input.

In [ ]:
_, turn = latest_turn()
retrieval_operations = turn["stages"]["03_retrieval"]["operations"]
hybrid_key = next(
    key for key in retrieval_operations
    if key.endswith("retrieval_hybrid_ranked")
)
hybrid_results = retrieval_operations[hybrid_key]["details"]["results"]

print(f"Trace section: {hybrid_key} (current line {trace_line(hybrid_key)})")
for result in hybrid_results:
    chunk = result["chunk"]
    location = (
        f"{chunk['path']}:{chunk['start_line']}-{chunk['end_line']}"
    )
    print(
        f"\nRank {result['rank']} | score={result['score']:.6f} | "
        f"{location}"
    )
    print(chunk["text_preview"])


## 9. Inspect the exact LLM input

`04_llm_input` shows the same request at three consecutive steps. Expand them in
order:

1. `01_structured_messages` — what KíLín prepares. This is a readable list of
   messages with roles. The `system` message contains KíLín's answering rules.
   The `user` message contains the question and the retrieved evidence. KíLín
   decides this content and organization.
2. `02_rendered_prompt` — the exact readable prompt after model formatting. The
   selected model's tokenizer owns a chat template. It joins the structured
   messages and adds model-specific markers that distinguish the system and user
   sections. Markers such as `<|im_start|>` are instructions understood by that
   model; they are not retrieval embeddings and should not be removed.
3. `03_input_token_ids` — the exact numerical input passed to the generation
   model. The tokenizer converts the rendered prompt into integer IDs. Each
   integer identifies a token in the model's vocabulary. These token IDs—not
   embedding vectors—are passed directly to the language model.

In short: question + evidence + instructions → structured messages → rendered
prompt text → token IDs → generated answer. The model ultimately receives token
IDs, but those IDs are simply the tokenized form of the rendered text shown
immediately above them.

The embedding model is used earlier only to find relevant chunks. Its vectors
help retrieval compare the question with project text; those vectors are never
included in the generation model's prompt. For a human audit, read
`01_structured_messages`, then `02_rendered_prompt`. Use `03_input_token_ids`
only when checking the tokenizer-level input.


In [ ]:
show_turn_stage("04_llm_input")


## 10. Inspect generation and saved output

`05_generation` is the result of the one LLM call shown in `04_llm_input`.
Inspect the model/backend, generation settings, exact output text, input and
output token counts, call status, and duration. Then inspect the saved turn and
checkpoint to confirm that the question, answer, evidence, settings, and lineage
can be resumed later. This answers: What did the model return, how expensive was
the call in tokens and time, and was the completed turn saved?


In [ ]:
show_turn_stage("05_generation")


## 11. Review the complete turn and its operation timings

The complete turn stays collapsed initially. Use it as a table of contents:
expand `01_question` through `05_generation` in order, or open only the stage
involved in a problem. The timing summary walks the same emitted operations and
lists only those that measured their own duration; it is useful for finding
whether embedding, retrieval, or generation was slow.


In [ ]:
turn_key, turn = latest_turn()
display_trace_tree(turn_key, turn)

print("\nOperation timings:")
for stage_key, stage in turn["stages"].items():
    for operation_key, operation in stage["operations"].items():
        duration = operation.get("timing", {}).get("operation_duration_ms")
        if duration is not None:
            print(f"{stage_key} / {operation_key}: {duration:,.3f} ms")


## 12. Close safely

The session releases both models. Because this notebook created and shared the
`TraceLogger` explicitly, it closes that logger after closing the session. The
final trace remains a self-contained JSON audit record inside the managed
project, and the final size printed below confirms that the file was written.


In [ ]:
session.close()
trace.close()

print(f"Final trace: {trace_path}")
print(f"Final trace size: {trace_path.stat().st_size / 1024:,.1f} KiB")


## Continue the tutorial series

**Previous:** [Tutorial 140 — Chat with a KíLín project](./140_project_chat.ipynb)

**Next:** [Tutorial 160 — Define project types and route intents](./160_project_types.ipynb)

The next tutorial introduces the configuration layer behind project
behavior, routing, and tool selection.
